# 05 - GRU4Rec

GRU4Rec compresses the whole trip prefix into a single recurrent hidden state and scores every city from that one state, instead of pairwise transitions (Markov/FPMC) or attending over the whole sequence (SASRec):

$$h_t = \text{GRU}(h_{t-1}, \, \text{emb}(c_t)) \qquad \text{score}(i) = h_T \cdot \text{emb}(i)$$

It is the first model in the project with genuine **long-range sequence memory** -- capability #3 on the sophistication ladder (Popularity -> Markov -> FPMC -> **GRU4Rec** -> SASRec). Trained with the same BPR next-item objective as SASRec, early-stopped on validation NDCG@10, so the two neural models are compared on equal footing.

All logic is in `src/models/gru4rec.py`; this notebook runs the side-feature ablation the plan calls for and logs results through the shared `evaluate.py`.

In [ ]:
import os, sys
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import torch
from src.preprocessing import load_processed, preprocess
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

## Ablation -- with vs without side features

Same trip-level context SASRec ablates (`device_class`, `booker_country`, `affiliate_id`), summed into the input embedding at every step here (the simpler of SASRec's two combination modes). Comparing with/without tells us whether that context helps a *recurrent* model the way -- or unlike the way -- it (didn't) help the attention-based one.

In [ ]:
from src.models.gru4rec import run_gru4rec, SIDE_FEATURES
from src.evaluate import append_result
RESULTS = os.path.join(ROOT, 'results.csv')

CONFIG = dict(dim=64, hidden=64, n_layers=1, dropout=0.2, residual=True,
              epochs=40, lr=1e-3, batch=256, patience=5, seed=42)
print('config:', CONFIG, '| device:', DEVICE)

for side_feats, tag in [((), 'no-side'), (SIDE_FEATURES, 'side')]:
    print(f'--- GRU4Rec ({tag}) ---')
    metrics, _, val = run_gru4rec(data, side_feats=side_feats, device=DEVICE,
                                  verbose=True, **CONFIG)
    print('best val NDCG@10:', round(val, 4))
    print('TEST:', {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    append_result(RESULTS, f'GRU4Rec-{tag}', metrics)

## What to write up

- GRU4Rec should beat Markov wherever a trip's *earlier* stops (not just the last one) carry predictive signal -- check the trip-length slicing in `06_comparison.ipynb` for direct evidence of this.
- Report the side-feature ablation result next to SASRec's (`05_sasrec.ipynb`): if trip-level context hurts both neural models similarly, that is a general finding about the dataset, not an architecture-specific quirk.
- Compute note: GRU4Rec is a single-layer recurrence over short trips, so it is comparable in cost to FPMC and much cheaper than SASRec -- see `n_params` / `train_secs` in `results.csv` for the accuracy-vs-cost discussion.